<a href="https://colab.research.google.com/github/AlexZhuo29/QCC_STEP_Research_2025_2026/blob/main/STEP_Research_Code_2025_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
## Libraries needed
import pandas as pd
import numpy as np

import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import csv
#import requests

## creating lag data (previous days)
def lag_func(dataframe, name, lag_day):
    ## creates a dataframe with just the Date and name column and assign it to df_lag
    df_lag = dataframe[['Date', name]].copy()

    ## start from day 1 to the lag_day provided b/c lag_day+1 is excluded
    for i in range(1, lag_day + 1):
        ## creates a new column and assign value into the column
        ## df_lag[name].shift(i) means to shift the previous value down
        df_lag[f'lag_{i}'] = df_lag[name].shift(i)

    ## removes empty spaces
    df_lag.dropna(inplace=True)

    ## return finished dataframe
    return df_lag

## process data and fit into dataframe
def process_func(name, symbol):
    ## create new empty dataframe
    df = pd.DataFrame()

    ## fill the dataframe with stock prices
    df[name] = yf.Ticker(symbol).history(start=START, end=END).Close

    ## add index
    df.reset_index(inplace=True)

    ## removes time
    df['Date'] = [i.date() for i in df.Date]

    ## adds a column for the stock price going up or down
    df['Up/Down'] = (df[name].shift(-1) > df[name]).astype(int)

    ## return dataframe
    return df

## Data Preparation
def prepare_func(name, symbol, lag_day):
  ## create a processed dataframe
  df = process_func(name, symbol)

  ## create a dataframe with the lag data
  df_lag = lag_func(df, name, lag_day)
  df_lag.head().round(4)

  ## update the lag dataframe's Up/Down with processed dataframe's Up/Down values at the same indices
  df_lag['Up/Down'] = df['Up/Down'].loc[df_lag.index]

  ## drop the empty cells
  df_lag.dropna(inplace=True)

  ## return dataframe
  return df_lag

## define input and output
def defineXY_func(dataframe): ## x is input and y is output
  ## lag data is the input
  x = dataframe[[f'lag_{i}' for i in range(1, 6)]]
  ## output is whether the stock went up or down
  y = dataframe['Up/Down']

  ## shape the number of x and y
  print(x.shape)
  print(y.shape)

  ## return input and output
  return x,y

## split data for training and testing
def splitData_func(x,y):
  ## initialize the scaler tool which processes the data so it will be compatiable with KNN
  scaler = StandardScaler()

  ## scaler figures out scaling parameters of x and apply it to the input
  x_scaled = scaler.fit_transform(x)

  ## splitting data into 80 / 20. 80% of the dataset are used for training and 20% used for testing
  split = int(len(x_scaled) * 0.8)

  ## assign front 80% to training data and remaining 20% to testing data
  x_train, x_test = x_scaled[:split], x_scaled[split:]
  y_train, y_test = y.iloc[:split], y.iloc[split:]

  ## return the assigned dataset
  return x_train, x_test, y_train, y_test

## 5 year time frame
START = "2010-1-1"
END = "2015-12-31"

## companies that will be tested
company_name = ['Google', 'Oracle', 'JP Morgan', 'Johnson and Johnson', 'Delta', 'Costco']
company = ['GOOGL', 'ORCL', 'JPM', 'JNJ', 'DAL', 'COST']

In [78]:
df_GOOG = prepare_func(company_name[0], company[0], 5)
x_GOOG,y_GOOG = defineXY_func(df_GOOG)
x_GOOG_train, x_GOOG_test, y_GOOG_train, y_GOOG_test = splitData_func(x_GOOG,y_GOOG)

(1504, 5)
(1504,)


# KNN

In [ ]:
# KNN Classifier
from sklearn.neighbors import KNeighborsClassifier

# DataFrame declaration and initialization
KNNC_df = pd.DataFrame(columns = ['N_neighbors', 'Weights', 'Algorithm', 'Leaf Size', 'Power', 'Train Score', 'Test Score'])

# Parameter Set
KNN_algorithm = {"auto", "ball_tree", "kd_tree", "brute"}
KNN_weights = {None, "uniform", "distance"}


# Model (n_neighbors)
for NNeighbors in range(1,100,10):
  for Weights in KNN_weights:
    for Algorithm in KNN_algorithm:
      for LeafSize in range(1,100,10):
        for Power in np.arange(1,100,10): # (42,45) (53,55) (55,57)
          knn = KNeighborsClassifier(n_neighbors = NNeighbors, weights = Weights, algorithm = Algorithm, leaf_size = LeafSize, p = Power)
          knn.fit(x_train, y_train)

          # Prediction
          knn.predict(x_train[:5]), knn.predict(x_test[:5])

          # Score
          trainScore = knn.score(x_train, y_train)
          testScore = knn.score(x_test, y_test)

          # Update DataFrame
          KNNC_df.loc[len(KNNC_df)] = [NNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore]

          if(testScore >= 0.6):
            key = False
            print(NNeighbors, Weights, Algorithm, LeafSize, Power, trainScore, testScore)



In [ ]:
KNNC_df.round(2).sort_values('Test Score', ascending=False)

In [ ]:
'''from google.colab import drive
drive.mount('/content/drive')
KNNC_df.to_csv('/content/drive/My Drive/KNNClass1.csv', index=False)'''